# 🫁 Pulmonology Chest X-Ray Classification: Pipeline 1 (DenseNet121 Baseline)

## 📌 Overview
This notebook demonstrates **Pipeline 1**, an end-to-end deep learning pipeline for 4-class medical Chest X-ray classification:
- **Classes:** `COVID-19`, `Normal`, `Lung Opacity`, `Viral Pneumonia`
- **Model:** `DenseNet121` with Transfer Learning
- **Loss Function:** `CrossEntropyLoss`
- **Learning Rate Scheduler:** `ReduceLROnPlateau`
- **Engine:** Custom `BaseTrainer` with AMP, Gradient Accumulation, and Checkpointing


### Step 1: Import Dependencies & Path Setup

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torchvision import transforms
import pandas as pd
from PIL import Image
from pathlib import Path

# Add project root directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from core.models.pulmonology.pipeline_1.data_engineering import get_dataloaders
from core.models.pulmonology.pipeline_1.model_engineering import get_densenet_model
from core.engine.base_trainer import BaseTrainer, create_experiment_dir, Logger

print(f"PyTorch Version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Active Device: {device.upper()}")

: 

### Step 2: Data Engineering & DataLoader Setup
We load images using standard dataset splits (`train.csv`, `validation.csv`) and apply standard baseline augmentations.

In [ ]:
splits_dir = '../../../dataset/splits/pulmonology/chest_xray'
images_dir = '../../../dataset/processed/pulmonology/chest_xray/images'
batch_size = 32

train_loader, val_loader = get_dataloaders(splits_dir, images_dir, batch_size=batch_size)

print(f"Training Batches: {len(train_loader)}")
print(f"Validation Batches: {len(val_loader)}")

### Step 3: Model Architecture (DenseNet121)
DenseNet121 is initialized with ImageNet pre-trained weights, and the final classification head is modified to output 4 classes.

In [ ]:
model = get_densenet_model(num_classes=4, pretrained=True)
print(model.classifier)

### Step 4: Loss Function, Optimizer & Scheduler
- Loss: Standard `CrossEntropyLoss`
- Optimizer: `Adam` (LR = 1e-4)
- Scheduler: `ReduceLROnPlateau` (halves LR if validation loss doesn't improve for 2 epochs)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

### Step 5: Training Execution (BaseTrainer)
Launch the training loop with early stopping, mixed precision, and dynamic experiment directory tracking.

In [ ]:
exp_dir = create_experiment_dir(module_name="pulmonology", model_name="DenseNet121", base_path="../../../core/models")

trainer = BaseTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    exp_dir=exp_dir,
    patience=5,
    accumulation_steps=1
)

# Fit the model for 10 epochs
# trainer.fit(num_epochs=10, resume=False)